# Module 10: Processing JSON and Arrays in PostgreSQL

**ALY 6420 | JSON, JSONB, arrays, extraction, expansion, flattening, and design decisions**

*Course Lecture Notes*


## Module 10

### Moving beyond flat relational columns

The relational model works best when each attribute has a stable meaning and each cell contains one atomic value.

Real systems often contain data that is less regular:

```text
product specifications
API payloads
event metadata
variable-length tag lists
customer preferences
nested transaction details
```

PostgreSQL supports these patterns with **JSON/JSONB** and **array** data types.

This module focuses on two skills:

1. querying these complex values correctly, and
2. recognizing when they are appropriate versus when ordinary relational tables are the better design.


## Learning objectives

By the end of this lecture, you should be able to:

- distinguish structured, semi-structured, and unstructured data
- explain why JSON appears in modern analytical systems
- distinguish PostgreSQL `json` from `jsonb`
- use `->` and `->>` correctly
- navigate nested JSON structures
- cast extracted JSON text into numeric or date/time types
- use JSON containment and path functions
- expand JSON objects and arrays into relational rows
- explain row multiplication caused by expansion functions
- construct and query PostgreSQL arrays
- use `ANY`, `ALL`, `@>`, and `&&`
- use `array_agg`, `unnest`, `cardinality`, `array_length`, and `array_to_string`
- flatten semi-structured values into analysis-ready tables
- combine JSON extraction with CTEs, aggregation, and filtering
- evaluate when JSON, arrays, or normalized relations are the better architecture
- identify common correctness and performance problems in semi-structured SQL


## Principal source and database

### Principal source

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.), **Chapter 11: Processing JSON and Arrays**.

The textbook develops the topic by moving from:

1. structured versus semi-structured data,
2. JSON representation,
3. JSONB storage,
4. extracting and searching JSON values,
5. JSON path,
6. flattening nested JSON,
7. array construction and aggregation, and
8. reversing arrays back into rows.

### Working database

Beginning in this module, examples are oriented toward the **ZoomZoom / `sqlda`** database imported in Module 9.

Because dataset versions can differ slightly, inspect your own schema before assuming a particular complex-data column name.


## First step: inspect the schema

Use metadata rather than guessing:

```sql
SELECT
    table_name,
    column_name,
    data_type,
    udt_name
FROM information_schema.columns
WHERE table_schema = 'public'
  AND (
      data_type IN ('json', 'jsonb', 'ARRAY')
      OR udt_name LIKE '\_%'
  )
ORDER BY table_name, column_name;
```

Record the actual JSON/JSONB and array columns in your imported `sqlda` instance.


# Part 1: Structured, Semi-Structured, and Unstructured Data


## Structured data

Structured data has a known schema.

Example:

```text
customer_id  INTEGER
first_name   TEXT
state        TEXT
date_added   TIMESTAMP
```

The database knows:

- which fields exist,
- their types,
- how rows and columns are organized,
- and which constraints apply.


## Semi-structured data

Semi-structured data contains labels or structure, but that structure may vary from record to record.

A JSON example:

```json
{
  "model": "Blade",
  "range_km": 80,
  "features": ["GPS", "ABS"]
}
```

Another record might contain additional keys or omit some entirely.


## Unstructured data

Unstructured data does not expose a convenient database schema directly.

Examples include:

- audio
- video
- arbitrary images
- free-form documents

SQL is primarily designed for structured data and can work with selected semi-structured formats when the structure or path you need is known.


## Quick check 1

**Think before opening the answer.**

Why is JSON called semi-structured rather than completely unstructured?

<details>
<summary>Answer</summary>

Because JSON values still carry structure through keys, nested objects, arrays, and typed-looking values, even though every record does not need to have exactly the same set or order of fields.

</details>


## A key limitation

SQL can query JSON efficiently when you know what you are looking for.

For example, if you know the path:

```text
customer → sales → product_name
```

you can extract it.

If you do not know:

- which keys exist,
- how deeply they are nested,
- or what shape the structure takes,

the problem becomes much harder.

This is why semi-structured flexibility does not eliminate the need to understand schema.


# Part 2: Why JSON Exists in Relational Databases


## Real-world variability

Imagine a product catalog.

One scooter might have:

```text
battery_capacity
range
max_speed
charging_time
```

Another product might also include:

```text
cargo_capacity
motor_count
drive_mode
weather_rating
```

A single fixed relational table can become awkward if every product has a very different set of attributes.


## JSON as key-value representation

JSON represents values using labeled keys:

```json
{
  "product_id": 5,
  "model": "Blade",
  "performance": {
    "range_km": 80,
    "max_speed_kph": 45
  }
}
```

Values may be:

- strings
- numbers
- Boolean values
- `null`
- objects
- arrays


## JSON flexibility is not free

Advantages:

- variable attributes
- nested structures
- easy interchange with APIs
- natural representation of event payloads

Costs:

- more complex queries
- explicit type conversion
- weaker discoverability for analysts
- repeated extraction logic
- possible performance overhead


## PostgreSQL can create JSON from rows

The textbook demonstrates `row_to_json`:

```sql
SELECT row_to_json(c)
FROM customers c
LIMIT 1;
```

This turns one relational row into one JSON document using column names as keys.


## Pretty-print JSON for inspection

```sql
SELECT row_to_json(c, TRUE)
FROM customers c
LIMIT 1;
```

Pretty formatting is useful for human inspection.

It does not change the underlying analytical meaning of the values.


# Part 3: JSON versus JSONB


## PostgreSQL supports two JSON storage types

### `json`

Stores the original JSON text representation.

### `jsonb`

Stores a parsed binary representation optimized for PostgreSQL querying.


## Why JSONB is commonly preferred for analytics

The textbook emphasizes that ordinary JSON must be parsed to interpret its keys and values.

`jsonb` performs parsing at storage time, making later key/value access more efficient.

`jsonb` also supports additional operators and indexing strategies that make it especially useful for analytical workloads.


## Important nuance

`jsonb` is not identical to raw JSON text preservation.

Because it stores a parsed representation:

- original whitespace is not meaningful
- key order should not be treated as preserved
- duplicate keys are not a safe design pattern

If byte-for-byte preservation matters, raw `json` or original source storage may be more appropriate.


## Quick check 2

**Think before opening the answer.**

For repeated analytical filtering and extraction, which PostgreSQL type is usually the stronger default: `json` or `jsonb`?

<details>
<summary>Answer</summary>

`jsonb`, because it stores a parsed representation and supports richer querying and indexing.

</details>


# Part 4: The `->` Operator


## Extract a JSON value and preserve JSON type

```sql
SELECT
    '{"a": 1, "b": 2}'::jsonb -> 'b' AS b_value;
```

The `->` operator returns a JSON/JSONB value.

Use it when:

- you need to keep navigating,
- the result is still a nested object or array,
- or you want to operate on the value as JSON.


## Example with nested structure

```sql
SELECT
    '{
       "model": "Blade",
       "performance": {
         "range_km": 80
       }
     }'::jsonb
     -> 'performance' AS performance_object;
```

The result is still a JSON object:

```json
{"range_km": 80}
```


# Part 5: The `->>` Operator


## Extract as text

```sql
SELECT
    '{"a": 1, "b": 2}'::jsonb ->> 'b' AS b_text;
```

`->>` returns **text**.

This is usually what you want for:

- display
- string comparison
- grouping
- casting into a SQL numeric/date type


## `->` versus `->>`

| Operator | Returns | Typical use |
|---|---|---|
| `->` | JSON/JSONB | keep navigating |
| `->>` | text | final scalar extraction |


## A reliable nested rule

For a nested path:

```sql
json_column
    -> 'outer_object'
    -> 'inner_object'
    ->> 'final_scalar'
```

Use `->` while moving through the structure.

Use `->>` when you want the final scalar as SQL text.


## Quick check 3

**Think before opening the answer.**

You need to extract `performance.max_speed_kph` and compare it numerically. Should the final operator be `->` or `->>`?

<details>
<summary>Answer</summary>

`->>` should normally be the final extraction so the value becomes text, after which you can cast it to an integer or numeric type.

</details>


# Part 6: Casting Extracted JSON Values


## `->>` returns text even when the JSON value looks numeric

Suppose JSON contains:

```json
{"max_speed_kph": 45}
```

This expression:

```sql
specs ->> 'max_speed_kph'
```

returns SQL text.

For arithmetic:

```sql
(specs ->> 'max_speed_kph')::INTEGER
```


## Why explicit casting matters

Text ordering and numeric ordering are different.

Text:

```text
100
20
3
```

can behave differently from:

```text
3
20
100
```

in numeric comparison.

If a JSON scalar represents a number, convert it into a numeric SQL type before mathematical analysis.


## Nested numeric extraction

```sql
SELECT
    (
        specs
        -> 'performance'
        ->> 'max_speed_kph'
    )::INTEGER AS max_speed_kph
FROM products;
```

Always validate that all values at the path can actually be cast.


## Defensive casting workflow

Before casting a newly discovered JSON field:

```sql
SELECT DISTINCT
    specs -> 'performance' ->> 'max_speed_kph'
FROM products
ORDER BY 1;
```

Inspect the values first.

Look for:

```text
N/A
unknown
blank strings
unexpected units
```


# Part 7: Nested Paths


## Chaining operators

Short path:

```sql
SELECT
    specs -> 'performance' ->> 'range_km'
FROM products;
```

This reads naturally:

```text
specs
  → performance
      → range_km
```


## JSON array indexing is zero-based

Example:

```sql
SELECT
    '["red", "blue", "green"]'::jsonb -> 0 AS first_json,
    '["red", "blue", "green"]'::jsonb ->> 0 AS first_text;
```

Index `0` is the first element.


## `#>` for path arrays

The textbook also introduces path access using `#>`:

```sql
SELECT
    '{
      "a": 1,
      "b": [{"d": 4}, {"d": 6}]
    }'::jsonb
    #> ARRAY['b', '1', 'd'];
```

The array on the right describes the navigation path.


## `jsonb_extract_path_text`

Another readable option:

```sql
SELECT
    jsonb_extract_path_text(
        specs,
        'performance',
        'max_speed_kph'
    ) AS max_speed_kph
FROM products;
```

This is especially useful when the path is long and you only need the final text value.


## Quick check 4

**Think before opening the answer.**

In a JSON array, does index `1` refer to the first or second element?

<details>
<summary>Answer</summary>

The second element. PostgreSQL JSON array indexing is zero-based.

</details>


# Part 8: JSON Containment


## Search by contained key-value pairs

For `jsonb`, the containment operator `@>` can test whether the left JSON contains a structure on the right.

```sql
SELECT *
FROM customer_sales
WHERE customer_json
      @> '{"customer_id": 20}'::jsonb;
```

This asks:

> Does this JSONB document contain `customer_id = 20`?


## Containment is structural

This is different from searching a raw text string.

You are comparing JSON structure, not merely asking whether the characters `"20"` occur somewhere.


## Containment can support indexing

A GIN index on a `jsonb` column can help PostgreSQL answer many containment/existence queries more efficiently.

Example design pattern:

```sql
CREATE INDEX idx_customer_sales_json
ON customer_sales
USING GIN (customer_json);
```

Index creation is a schema design decision; understand the workload before adding one.


# Part 9: JSON Path


## JSON path gives another way to navigate

The textbook introduces PostgreSQL's JSON path language.

A path begins at:

```text
$
```

meaning the JSON root.

Example:

```sql
$.sales[0]
```

means:

```text
root
 → sales
 → first array element
```


## Check whether a path exists

```sql
SELECT
    jsonb_path_exists(
        customer_json,
        '$.sales[0]'
    )
FROM customer_sales;
```

This returns a Boolean.


## Query a path

```sql
SELECT
    jsonb_path_query(
        customer_json,
        '$.sales[0].sales_amount'
    )
FROM customer_sales;
```

This retrieves matching JSONB values.


## Wildcard path

```sql
$.sales[*].sales_amount
```

means:

> all `sales_amount` values from all elements of the `sales` array.


## Path filtering

A path can include conditions:

```sql
SELECT
    jsonb_path_query_array(
        customer_json,
        '$.sales[*].sales_amount ? (@ > 400)'
    )
FROM customer_sales;
```

This returns an array of matching values.


## Important row-count behavior

Some set-returning JSON path functions can produce:

- multiple rows,
- one row,
- or no row

depending on how many matches exist.

Do not assume one source row always produces one result row.


## Quick check 5

**Think before opening the answer.**

Why should you check row counts after using `jsonb_path_query` or expansion functions?

<details>
<summary>Answer</summary>

Because one input JSON document can produce zero, one, or multiple output rows, changing the grain of the result.

</details>


# Part 10: Expanding JSON Objects


## From one object to key-value rows

A JSON object:

```json
{
  "range_km": 80,
  "max_speed_kph": 45,
  "charge_hours": 4
}
```

can be converted into rows:

```text
range_km        80
max_speed_kph   45
charge_hours    4
```


## `jsonb_each`

```sql
SELECT
    kv.key,
    kv.value
FROM jsonb_each(
    '{
       "range_km": 80,
       "max_speed_kph": 45
     }'::jsonb
) AS kv(key, value);
```

`value` remains JSONB.


## `jsonb_each_text`

```sql
SELECT
    kv.key,
    kv.value
FROM jsonb_each_text(
    '{
       "range_km": 80,
       "max_speed_kph": 45
     }'::jsonb
) AS kv(key, value);
```

The values are returned as text.


## Expansion against a table

```sql
SELECT
    p.product_id,
    p.model,
    kv.key,
    kv.value
FROM products p
CROSS JOIN LATERAL
    jsonb_each(p.specs) AS kv(key, value);
```

Each product can generate several key-value rows.


## Grain changes during JSON expansion

Before expansion:

```text
one row per product
```

After `jsonb_each`:

```text
one row per product × JSON key
```

This is conceptually similar to a one-to-many join.


# Part 11: Expanding JSON Arrays


## JSON arrays need a different expansion function

Suppose:

```json
{
  "regions": ["Northeast", "Midwest", "West"]
}
```

Use:

```sql
jsonb_array_elements(...)
```

or:

```sql
jsonb_array_elements_text(...)
```


## Elements as JSONB

```sql
SELECT
    elem
FROM jsonb_array_elements(
    '["Northeast", "Midwest", "West"]'::jsonb
) AS elem;
```


## Elements as text

```sql
SELECT
    region
FROM jsonb_array_elements_text(
    '["Northeast", "Midwest", "West"]'::jsonb
) AS region;
```

For simple strings, the text variant is usually easier.


## Nested array of objects

Suppose each customer JSON contains:

```json
"sales": [
  {
    "product_id": 5,
    "product_name": "Blade",
    "sales_amount": 559.99
  },
  {
    "product_id": 8,
    "product_name": "Lemon",
    "sales_amount": 399.99
  }
]
```

Each sale can be expanded into its own row.


## Textbook-style sales expansion

```sql
SELECT
    customer_json,
    jsonb_array_elements(
        customer_json -> 'sales'
    ) AS sale_json
FROM customer_sales;
```

The result grain becomes:

> one row per customer sale element


## Extract fields after expansion

```sql
WITH expanded_sales AS (
    SELECT
        customer_json,
        jsonb_array_elements(
            customer_json -> 'sales'
        ) AS sale_json
    FROM customer_sales
)
SELECT
    customer_json ->> 'first_name' AS first_name,
    customer_json ->> 'last_name'  AS last_name,
    sale_json ->> 'product_name'   AS product_name,
    (sale_json ->> 'sales_amount')::NUMERIC AS sales_amount
FROM expanded_sales;
```


## Why a CTE helps

Nested JSON queries become difficult when:

- expansion,
- extraction,
- casting,
- filtering,
- and aggregation

are all written in one SELECT.

A CTE lets you name the intermediate grain:

```text
one row per sale
```

before adding more logic.


# Part 12: JSON Fan-Out and Missing Rows


## Expansion is fan-out

If one customer has five sales:

```text
1 customer row
    ↓ jsonb_array_elements(sales)
5 sale rows
```

Aggregates after the expansion must be interpreted at the new grain.


## Empty arrays produce zero child rows

If:

```json
"sales": []
```

then expanding `sales` produces no sale element.

The parent customer may disappear from the expanded result unless you deliberately preserve it.


## Preserve parents with LEFT JOIN LATERAL

Pattern:

```sql
SELECT
    c.customer_json,
    sale.sale_json
FROM customer_sales c
LEFT JOIN LATERAL
    jsonb_array_elements(
        c.customer_json -> 'sales'
    ) AS sale(sale_json)
ON TRUE;
```

This can preserve a customer even when the sales array is empty.


## Why this matters

Suppose the question is:

> What percentage of customers made no purchase?

A plain cross expansion removes customers with no sale elements.

That makes the denominator wrong.

Grain and row preservation still matter with semi-structured data.


## Quick check 6

**Think before opening the answer.**

If a customer has an empty JSON sales array, what happens with a plain `jsonb_array_elements` cross expansion?

<details>
<summary>Answer</summary>

The customer produces zero child rows and therefore disappears from the expanded result.

</details>


# Part 13: PostgreSQL Arrays


## Arrays store multiple values of one base type

Example:

```sql
SELECT
    ARRAY['Lemon', 'Blade', 'Bat']
    AS purchased_products;
```

PostgreSQL infers this as a text array.


## Arrays are homogeneous

A PostgreSQL array contains values of a compatible base type.

Examples:

```text
INTEGER[]
TEXT[]
NUMERIC[]
DATE[]
```

This differs from JSON arrays, which can contain more heterogeneous nested structures.


## Array indexing

PostgreSQL native arrays normally use **1-based indexing**.

This is different from JSON arrays, whose extraction indexes are zero-based.

That distinction is easy to forget.


## JSON array versus PostgreSQL array

```text
JSON array index:        0, 1, 2, ...
PostgreSQL array index:  1, 2, 3, ...
```

Be deliberate about which data type you are working with.


## Quick check 7

**Think before opening the answer.**

What is the first index of a normal PostgreSQL array?

<details>
<summary>Answer</summary>

1. PostgreSQL native arrays are ordinarily 1-based, unlike JSON arrays, which are zero-based.

</details>


# Part 14: Building Arrays with `array_agg`


## Aggregate rows into one array

The textbook demonstrates:

```sql
SELECT
    product_type,
    ARRAY_AGG(DISTINCT model) AS models
FROM products
GROUP BY product_type;
```

The result has:

> one row per product type, with an array of models


## Order elements inside the array

```sql
SELECT
    product_type,
    ARRAY_AGG(
        model
        ORDER BY year
    ) AS models
FROM products
GROUP BY product_type;
```

The `ORDER BY` belongs inside the aggregate because it controls the element order in the created array.


## DISTINCT inside `array_agg`

```sql
ARRAY_AGG(DISTINCT model ORDER BY model)
```

can be useful when the goal is a set-like list of unique labels.

Only use `DISTINCT` when duplicates are not analytically meaningful.


# Part 15: Reversing Arrays with `unnest`


## `unnest` converts array elements to rows

```sql
SELECT
    UNNEST(ARRAY[123, 456, 789]) AS example_id;
```

Result:

```text
123
456
789
```


## Expand an array column

```sql
SELECT
    c.customer_id,
    tag
FROM customers c
CROSS JOIN LATERAL
    unnest(c.interest_tags) AS tag;
```

Result grain:

> one row per customer × array element


## Count by array element

```sql
SELECT
    tag,
    COUNT(*) AS customer_tag_rows
FROM customers c
CROSS JOIN LATERAL
    unnest(c.interest_tags) AS tag
GROUP BY tag
ORDER BY customer_tag_rows DESC;
```


## Check the analytical meaning of the count

If the same tag appears twice in one customer's array, then:

```sql
COUNT(*)
```

counts both element rows.

If the question is "how many customers have this tag?" consider:

```sql
COUNT(DISTINCT customer_id)
```

after expansion.


# Part 16: Array Membership


## `= ANY(array)`

```sql
SELECT
    'Blade' = ANY(
        ARRAY['Lemon', 'Blade', 'Bat']
    ) AS has_blade;
```

Returns `TRUE` if the value equals at least one array element.


## `= ALL(array)`

```sql
SELECT
    10 > ALL(ARRAY[1, 4, 9])
    AS greater_than_every_value;
```

`ALL` requires the comparison to be true for every element.


## Array containment `@>`

```sql
SELECT
    ARRAY['EV', 'Cargo', 'Luxury']
    @> ARRAY['EV', 'Cargo']
    AS contains_both;
```

The left array must contain all elements of the right.


## Array overlap `&&`

```sql
SELECT
    ARRAY['EV', 'Cargo']
    && ARRAY['Luxury', 'Cargo']
    AS overlaps;
```

Returns true if the arrays share at least one element.


## Quick check 8

**Think before opening the answer.**

Which operator would you use to ask whether an array contains both `'EV'` and `'Cargo'`?

<details>
<summary>Answer</summary>

`@>` with a right-hand array containing both requested elements.

</details>


## Quick check 9

**Think before opening the answer.**

Which operator would you use to ask whether two arrays share at least one element?

<details>
<summary>Answer</summary>

`&&`.

</details>


# Part 17: Array Utility Functions


## `cardinality`

```sql
SELECT
    cardinality(
        ARRAY['A', 'B', 'C']
    ) AS element_count;
```

Returns the total number of elements.


## `array_length`

```sql
SELECT
    array_length(
        ARRAY['A', 'B', 'C'],
        1
    ) AS first_dimension_length;
```

The second argument specifies the dimension.


## `array_to_string`

The textbook demonstrates converting an array back into readable text:

```sql
SELECT
    array_to_string(
        ARRAY['Lemon', 'Bat Limited Edition'],
        ', '
    ) AS models;
```


## `string_to_array`

Reverse direction:

```sql
SELECT
    string_to_array(
        'hello world',
        ' '
    );
```

Result:

```text
{hello,world}
```


## Other useful array operations

Examples include:

```sql
array_append(arr, value)
array_cat(arr1, arr2)
array_remove(arr, value)
array_position(arr, value)
```

These are useful for array construction and manipulation, but query design should still begin with the business question.


# Part 18: NULL Arrays


## NULL array is different from empty array

```text
NULL
```

means the array value is unknown/missing.

```text
{}
```

means the array exists and contains zero elements.


## `unnest(NULL)` produces no element rows

That means the parent row disappears in a cross expansion.

If the parent must be preserved, use an outer lateral join pattern.


## Do not confuse row preservation with COALESCE

This:

```sql
unnest(COALESCE(tags, '{}'))
```

converts NULL to an empty array.

But an empty array still produces zero child rows in a cross join.

To truly preserve the parent row, use:

```sql
LEFT JOIN LATERAL ...
ON TRUE
```

when that is required by the business question.


# Part 19: Extract → Cast → Aggregate


## A standard analytical pattern

Semi-structured analytics often follows:

```text
extract
   ↓
cast
   ↓
filter / clean
   ↓
aggregate
```

Each stage makes the next one simpler.


## CTE pattern

```sql
WITH extracted AS (
    SELECT
        specs ->> 'category' AS category,
        (
            specs
            -> 'performance'
            ->> 'max_speed_kph'
        )::INTEGER AS max_speed_kph
    FROM products
    WHERE specs IS NOT NULL
)
SELECT
    category,
    COUNT(*) AS product_count,
    AVG(max_speed_kph) AS avg_max_speed
FROM extracted
GROUP BY category
ORDER BY product_count DESC;
```


## Why this pattern is strong

Benefits:

- long JSON expressions appear once
- casts are explicit
- intermediate output is testable
- the outer query reads like ordinary relational SQL
- debugging is easier


## Inspect the extracted CTE first

Before aggregating:

```sql
WITH extracted AS (
    ...
)
SELECT *
FROM extracted
LIMIT 50;
```

Confirm:

- extracted values look right
- types are appropriate
- NULLs are understood
- units are consistent


# Part 20: Flattening into Relational Form


## Semi-structured data often becomes tabular before analysis

Suppose a product JSON contains variable attributes.

A flattened result may look like:

```text
product_id | attribute_name | attribute_value
-----------+----------------+----------------
5          | range_km       | 80
5          | max_speed_kph  | 45
7          | range_km       | 60
```

This is much easier to aggregate.


## Flatten with `jsonb_each`

```sql
SELECT
    p.product_id,
    p.model,
    kv.key AS attribute_name,
    kv.value AS attribute_value
FROM products p
CROSS JOIN LATERAL
    jsonb_each(p.specs) AS kv(key, value);
```


## Filter by JSON type

```sql
SELECT
    p.product_id,
    kv.key,
    kv.value
FROM products p
CROSS JOIN LATERAL
    jsonb_each(p.specs) AS kv(key, value)
WHERE jsonb_typeof(kv.value) = 'number';
```

This identifies attributes that can plausibly participate in numeric analysis.


## Numeric aggregation after flattening

```sql
SELECT
    kv.key AS attribute_name,
    AVG((kv.value::text)::NUMERIC) AS avg_value,
    COUNT(*) AS observations
FROM products p
CROSS JOIN LATERAL
    jsonb_each(p.specs) AS kv(key, value)
WHERE jsonb_typeof(kv.value) = 'number'
GROUP BY kv.key
ORDER BY avg_value DESC;
```

Be certain that the same key represents the same unit across products.


# Part 21: JSON/Array Design Decisions


## JSON is appropriate when variability is real

JSONB is often reasonable when:

- attributes vary significantly among rows
- payloads come from external APIs
- schema evolves frequently
- nested structure is natural
- the application often needs the whole object


## JSON is less attractive when fields are heavily queried

A normalized column/table is often better when:

- analysts repeatedly filter on the same field
- the field is used in joins
- the field is used in GROUP BY constantly
- the attribute has stable meaning and type
- many consumers repeat the same extraction


## Arrays are appropriate for compact homogeneous lists

Arrays can be useful for:

- tags
- small preference lists
- fixed-type codes
- compact ordered value collections

Arrays are less attractive when each element needs:

- its own attributes
- its own lifecycle
- foreign-key relationships
- frequent independent updates


## Relational normalization remains powerful

If each product can have many certifications and each certification has:

```text
certification_id
issuer
issue_date
expiration_date
status
```

a normalized child table is probably stronger than an array of certification names.


## A practical architecture test

Ask:

> Is this value truly one attribute of the parent, or is it actually another entity with its own attributes and relationships?

If it is another entity, a separate table is often the better model.


## Quick check 10

**Think before opening the answer.**

If an attribute is extracted from JSON in dozens of reports and is frequently filtered, joined, and grouped, what design question should you raise?

<details>
<summary>Answer</summary>

Whether that attribute should become a normal relational column or derived table instead of remaining buried inside JSON.

</details>


# Part 22: JSON and Array Performance Awareness


## Extraction has computational cost

This expression:

```sql
specs ->> 'category'
```

must be evaluated for rows that PostgreSQL examines.

On a small table that may be trivial.

At much larger scale, frequently extracted fields deserve indexing or redesign consideration.


## GIN index for JSONB

Pattern:

```sql
CREATE INDEX idx_products_specs_gin
ON products
USING GIN (specs);
```

Useful for many containment and key-oriented JSONB operations.


## Functional index for one extracted field

```sql
CREATE INDEX idx_products_category
ON products ((specs ->> 'category'));
```

This can help repeated equality filtering on that extracted scalar.


## Arrays can also be indexed

GIN indexes can support many PostgreSQL array containment/overlap queries.

But indexes are not a substitute for good modeling.

If an array becomes a large frequently joined relationship, a normalized table may be cleaner and faster.


# Part 23: Common Errors


## Error 1: using `->` when text is needed

Problem:

```sql
WHERE specs -> 'category' = 'EV'
```

The left side is JSONB while the right side is text.

Better:

```sql
WHERE specs ->> 'category' = 'EV'
```


## Error 2: numeric comparison as text

Problem:

```sql
WHERE specs ->> 'range_km' > '100'
```

This is a text comparison.

Better:

```sql
WHERE (specs ->> 'range_km')::INTEGER > 100
```


## Error 3: wrong array indexing convention

JSON array:

```sql
json_col -> 0
```

Native PostgreSQL array:

```sql
arr[1]
```

Mixing the indexing systems creates off-by-one mistakes.


## Error 4: ignoring fan-out

Problem:

```sql
FROM products p,
     jsonb_each(p.specs)
```

then later assuming:

```text
one row per product
```

The expansion changed the grain.


## Error 5: missing parent rows

Plain `unnest` or `jsonb_array_elements` expansion may remove parent rows that have NULL or empty collections.

If those parent rows belong in the analytical population, use a preservation strategy such as `LEFT JOIN LATERAL`.


## Error 6: repeated extraction everywhere

If every query contains:

```sql
(specs -> 'performance' ->> 'max_speed_kph')::INTEGER
```

the code becomes harder to maintain.

Use:

- a CTE
- a view
- a derived table
- or schema redesign


## Error 7: assuming JSON key existence

This may return NULL:

```sql
specs ->> 'battery_kwh'
```

if the key is absent.

Missing key and explicit JSON `null` may require careful interpretation.


## Error 8: aggregating incompatible JSON values

Suppose the same key means:

```text
range in km
```

for some records and:

```text
range in miles
```

for others.

The values may cast successfully but the average is meaningless.

Type correctness does not guarantee semantic correctness.


# Part 24: Debugging Semi-Structured Queries


## Step 1: inspect a few raw values

```sql
SELECT
    product_id,
    specs
FROM products
WHERE specs IS NOT NULL
LIMIT 10;
```


## Step 2: pretty-print if needed

```sql
SELECT
    jsonb_pretty(specs)
FROM products
WHERE specs IS NOT NULL
LIMIT 5;
```


## Step 3: inspect keys

```sql
SELECT DISTINCT
    key
FROM products p
CROSS JOIN LATERAL
    jsonb_object_keys(p.specs) AS key
ORDER BY key;
```

This helps reveal what fields actually exist.


## Step 4: extract without casting

```sql
SELECT DISTINCT
    specs -> 'performance' ->> 'max_speed_kph'
FROM products;
```

Look for unexpected values.


## Step 5: cast and validate ranges

```sql
SELECT
    MIN((specs -> 'performance' ->> 'max_speed_kph')::INTEGER),
    MAX((specs -> 'performance' ->> 'max_speed_kph')::INTEGER)
FROM products;
```


## Step 6: measure row multiplication

Before expansion:

```sql
SELECT COUNT(*)
FROM products;
```

After expansion:

```sql
SELECT COUNT(*)
FROM products p
CROSS JOIN LATERAL
    jsonb_each(p.specs) AS kv(key, value);
```

The difference tells you how expansion changed the grain.


# Part 25: Guided Practice


## Practice 1: simple JSON extraction

Return the model and range from this literal:

```json
{
  "model": "X1 Pro",
  "performance": {
    "range_km": 60
  }
}
```

Return model as text and range as integer.


<details>
<summary>Solution</summary>

```sql
SELECT
    '{"model":"X1 Pro","performance":{"range_km":60}}'::jsonb
        ->> 'model' AS model,
    (
      '{"model":"X1 Pro","performance":{"range_km":60}}'::jsonb
        -> 'performance'
        ->> 'range_km'
    )::INTEGER AS range_km;
```

</details>


## Practice 2: JSON containment

Write a Boolean expression that checks whether this JSONB contains `"category": "Scooter"`.


<details>
<summary>Solution</summary>

```sql
SELECT
    '{"category":"Scooter","price":499}'::jsonb
    @> '{"category":"Scooter"}'::jsonb
    AS contains_category;
```

</details>


## Practice 3: expand an object

Convert this object into key-value rows:

```json
{"range_km":80,"speed_kph":45,"charge_hours":4}
```


<details>
<summary>Solution</summary>

```sql
SELECT
    key,
    value
FROM jsonb_each(
    '{"range_km":80,"speed_kph":45,"charge_hours":4}'::jsonb
);
```

</details>


## Practice 4: expand a JSON array

Turn:

```json
["Northeast","Midwest","West"]
```

into one text row per region.


<details>
<summary>Solution</summary>

```sql
SELECT region
FROM jsonb_array_elements_text(
    '["Northeast","Midwest","West"]'::jsonb
) AS region;
```

</details>


## Practice 5: build an array

Return one row per `product_type` with an alphabetically sorted array of distinct model names.


<details>
<summary>Solution</summary>

```sql
SELECT
    product_type,
    ARRAY_AGG(
        DISTINCT model
        ORDER BY model
    ) AS models
FROM products
GROUP BY product_type
ORDER BY product_type;
```

</details>


## Practice 6: unnest an array

Expand:

```sql
ARRAY['EV', 'Cargo', 'Commuter']
```

to one row per tag.


<details>
<summary>Solution</summary>

```sql
SELECT
    UNNEST(
        ARRAY['EV', 'Cargo', 'Commuter']
    ) AS tag;
```

</details>


## Practice 7: array overlap

Return whether these arrays have at least one common element:

```text
['EV', 'Cargo']
['Luxury', 'Cargo']
```


<details>
<summary>Solution</summary>

```sql
SELECT
    ARRAY['EV', 'Cargo']
    && ARRAY['Luxury', 'Cargo']
    AS overlaps;
```

</details>


## Practice 8: array containment

Return whether:

```text
['EV', 'Cargo', 'Luxury']
```

contains both:

```text
['EV', 'Cargo']
```


<details>
<summary>Solution</summary>

```sql
SELECT
    ARRAY['EV', 'Cargo', 'Luxury']
    @> ARRAY['EV', 'Cargo']
    AS contains_both;
```

</details>


# Part 26: Real-World Examples


## API event payloads

An application may store webhook payloads such as:

```json
{
  "event": "payment_completed",
  "account_id": 811,
  "metadata": {
    "campaign": "fall_launch",
    "channel": "email"
  }
}
```

JSONB is useful when providers add or remove optional metadata over time.


## Product specifications

A catalog can contain broad common columns:

```text
product_id
model
product_type
base_price
```

and a JSONB column for variable technical specs.

Frequently queried specs may later be promoted to normal columns.


## Customer tags

A small array might contain:

```text
{"EV","Commuter","Eco"}
```

Useful questions include:

- does the customer have a specific tag?
- how many tags are assigned?
- which tags are most common?


## Marketing touchpoints

Arrays can represent short ordered sequences of touchpoints:

```text
{email,sms,web}
```

But if each touchpoint needs:

```text
timestamp
campaign_id
cost
response
```

a normalized event table is much stronger.


# Part 27: AI Critique


## AI-generated JSON SQL needs type review

Always inspect:

- whether `->` or `->>` is correct
- whether extracted text is cast
- whether JSON arrays are zero-based
- whether native arrays are one-based
- whether expansion changes row grain
- whether missing/empty arrays remove parent rows
- whether the path really exists in your schema


## AI critique exercise 1

A model suggests:

```sql
SELECT *
FROM products
WHERE specs ->> 'range_km' > '80';
```

It says this filters products with numeric range above 80.

What is wrong?


<details>
<summary>Critique</summary>

`->>` returns text, so the comparison is textual rather than numeric. Cast the extracted value:

```sql
WHERE (specs ->> 'range_km')::INTEGER > 80
```

after validating the field values.

</details>


## AI critique exercise 2

A model says:

> "Use `specs -> 1` to get the second value in a PostgreSQL `text[]` array."

What is wrong?


<details>
<summary>Critique</summary>

`->` is a JSON extraction operator. A native PostgreSQL array uses bracket indexing such as `arr[2]` for the second element. JSON arrays and PostgreSQL arrays have different syntax and indexing conventions.

</details>


## AI critique exercise 3

A model expands customer sales with `jsonb_array_elements` and then reports:

```sql
COUNT(*) AS customer_count
```

What should you verify?


<details>
<summary>Critique</summary>

After expansion the grain is one row per sale element, not one row per customer. `COUNT(*)` counts expanded sale rows. If you need customers, consider `COUNT(DISTINCT customer_id)` or aggregate at customer grain.

</details>


# Part 28: Lab 7 Readiness


## Before starting Lab 7

You should be comfortable with:

- identifying JSON and array columns from `information_schema`
- distinguishing JSON from JSONB
- using `->` and `->>`
- casting extracted scalar values
- navigating nested JSON
- expanding JSON objects
- expanding JSON arrays
- using CTEs to separate extraction from aggregation
- constructing arrays
- using `ANY`, `ALL`, `@>`, and `&&`
- using `unnest`
- explaining how expansion affects grain
- deciding when normalized tables are preferable


## Lab validation habit

For every expansion query, write a comment:

```sql
-- Grain before expansion: one row per customer
```

then:

```sql
-- Grain after expansion: one row per customer sale
```

This makes later counts and sums easier to reason about.


## Lab-style challenge

Suppose `customer_sales.customer_json` contains:

- customer fields at the root
- a `sales` JSON array
- each sale contains `product_name` and `sales_amount`

Return:

- customer ID
- customer name
- product name
- sales amount

one row per sale.


<details>
<summary>One solution</summary>

```sql
WITH expanded AS (
    SELECT
        customer_json,
        jsonb_array_elements(
            customer_json -> 'sales'
        ) AS sale_json
    FROM customer_sales
)
SELECT
    (customer_json ->> 'customer_id')::INTEGER AS customer_id,
    customer_json ->> 'first_name' AS first_name,
    customer_json ->> 'last_name' AS last_name,
    sale_json ->> 'product_name' AS product_name,
    (sale_json ->> 'sales_amount')::NUMERIC AS sales_amount
FROM expanded;
```

</details>


# Part 29: Quiz 2 Connection


## Quiz 2 is cumulative

Module 10 introduces new material, while Quiz 2 reviews earlier topics.

Review:

- conditional logic
- missing values
- text cleanup
- aggregation
- `GROUP BY`
- `HAVING`
- window functions
- import/export


## Complex-data work still uses old skills

A realistic Module 10 query may combine:

```text
JSON extraction
     +
CAST
     +
CASE
     +
CTE
     +
GROUP BY
     +
HAVING
```

New SQL features rarely replace older ones. They extend the toolkit.


# Part 30: Knowledge Checks


## Knowledge check 1

**Think before opening the answer.**

Which PostgreSQL JSON type is stored in a parsed binary representation?

<details>
<summary>Answer</summary>

`jsonb`.

</details>


## Knowledge check 2

**Think before opening the answer.**

Which operator returns JSON/JSONB and is useful for continued navigation?

<details>
<summary>Answer</summary>

`->`.

</details>


## Knowledge check 3

**Think before opening the answer.**

Which operator returns text?

<details>
<summary>Answer</summary>

`->>`.

</details>


## Knowledge check 4

**Think before opening the answer.**

Why should numeric values extracted with `->>` often be cast?

<details>
<summary>Answer</summary>

Because `->>` returns text, and numeric comparison/arithmetic requires a numeric SQL type for correct behavior.

</details>


## Knowledge check 5

**Think before opening the answer.**

What does `jsonb_array_elements` do?

<details>
<summary>Answer</summary>

It expands a JSONB array into one result row per array element.

</details>


## Knowledge check 6

**Think before opening the answer.**

What happens to result grain when one product JSON object with ten keys is expanded using `jsonb_each`?

<details>
<summary>Answer</summary>

One product row can become ten key-value rows, so the result moves from product grain to product-by-key grain.

</details>


## Knowledge check 7

**Think before opening the answer.**

What does `array_agg` do?

<details>
<summary>Answer</summary>

It aggregates multiple row values into a PostgreSQL array.

</details>


## Knowledge check 8

**Think before opening the answer.**

What does `unnest` do?

<details>
<summary>Answer</summary>

It expands a PostgreSQL array into one row per element.

</details>


## Knowledge check 9

**Think before opening the answer.**

Which array operator tests overlap?

<details>
<summary>Answer</summary>

`&&`.

</details>


## Knowledge check 10

**Think before opening the answer.**

Which array operator tests whether the left array contains all elements of the right?

<details>
<summary>Answer</summary>

`@>`.

</details>


## Knowledge check 11

**Think before opening the answer.**

Are PostgreSQL native arrays normally zero-based?

<details>
<summary>Answer</summary>

No. They are ordinarily one-based.

</details>


## Knowledge check 12

**Think before opening the answer.**

What is a warning sign that a JSON attribute may belong in a relational column?

<details>
<summary>Answer</summary>

The same field is repeatedly extracted, filtered, joined, and grouped across many queries.

</details>


# Part 31: Concept Maps


## JSON extraction map

```text
JSONB DOCUMENT
     |
     +-- -> key --------> JSONB value
     |                     |
     |                     +-- -> key --> nested JSONB
     |                     |
     |                     +-- ->> key -> text
     |
     +-- #> path -------> JSONB value
     |
     +-- jsonb_path_* --> path-based result
```


## Semi-structured analytics map

```text
RAW JSON / ARRAY
       |
       v
inspect structure
       |
       v
extract / expand
       |
       v
cast values
       |
       v
validate grain
       |
       v
clean / filter
       |
       v
aggregate / join
       |
       v
ANALYSIS-READY TABLE
```


## Storage decision map

```text
Stable scalar attribute?
    → normal column

Repeated entity with its own attributes?
    → normalized child table

Variable nested object / external payload?
    → JSONB

Small homogeneous list?
    → array

Frequently queried JSON/array field?
    → reconsider normalization or derived structure
```


# Part 32: Module Summary


## Key takeaways

- PostgreSQL can analyze semi-structured data without abandoning SQL.
- JSON is flexible but raw JSON text requires parsing.
- JSONB stores a parsed representation and is usually more suitable for repeated analytical querying.
- `->` preserves JSON structure; `->>` returns text.
- Extracted scalar text should be cast before numeric/date analysis.
- JSON path provides another way to navigate and filter nested structures.
- `jsonb_each` and `jsonb_array_elements` flatten nested structures into rows.
- Expansion changes grain and can remove parent rows with empty collections.
- PostgreSQL arrays store multiple values of one compatible base type.
- `array_agg` creates arrays; `unnest` reverses them into rows.
- `ANY`, `ALL`, `@>`, and `&&` support array membership and relationship tests.
- JSON arrays and PostgreSQL arrays use different indexing conventions.
- CTEs make extract-cast-aggregate workflows easier to test.
- JSON and arrays are tools, not replacements for normalization.
- Repeated extraction of the same field is a signal to reconsider schema design.


## The question to carry forward

When you see a JSON object or array in PostgreSQL, ask:

> **What is the relational grain I need for analysis, and what transformation gets this semi-structured value to that grain without losing or duplicating meaning?**

That question connects complex data types back to the same analytical reasoning used throughout the course.


## Up next: Module 11

Module 11 moves into advanced text analysis.

You will work with:

- text processing
- tokenization
- PostgreSQL full-text search
- search vectors and queries
- analytical patterns over unstructured-like text fields

The core lesson continues: PostgreSQL provides specialized types and functions, but trustworthy analysis still depends on understanding structure, type, and grain.


## References

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for data analytics: Analyze data effectively, uncover insights and master advanced SQL for real-world applications* (4th ed.). Packt Publishing.

PostgreSQL Global Development Group. (n.d.). *JSON types*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *JSON functions and operators*. PostgreSQL 16 Documentation.

PostgreSQL Global Development Group. (n.d.). *Array functions and operators*. PostgreSQL 16 Documentation.
